# 📉 Ridge Regression: From Scratch Implementation ( For N- dimensions)
> **Goal:** Build a custom Ridge Regression class using the Normal Equation and compare it with Scikit-Learn's implementation on the Boston Housing Dataset.

---
## 🧪 Project Overview
Regularization is a technique used to prevent overfitting by adding a penalty term to the cost function. In this notebook, we move beyond built-in libraries to implement the math ourselves.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/arunjathari/bostonhousepricedata/Boston-house-price-data.csv


## 📂 1. Data Acquisition
We are using the **Boston House Price Dataset**. It contains 13 features affecting housing prices (MEDV).

In [2]:
df = pd.read_csv("/kaggle/input/datasets/arunjathari/bostonhousepricedata/Boston-house-price-data.csv")

In [3]:
df.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2


In [4]:
# Features (all columns except the last one)
X = df.drop('MEDV', axis=1)

# Target variable
y = df['MEDV']

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 📏 2. Baseline Model (Linear Regression)
Before applying regularization, we establish a baseline using standard `LinearRegression`. This helps us understand the error rate (MAE) and accuracy ($R^2$) without any penalties.

In [7]:
pipeline = Pipeline([
    ('scalar', StandardScaler()),
    ("model", LinearRegression())]
)

In [8]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('scalar', StandardScaler()), ('model', LinearRegression())])

In [9]:
y_pred = pipeline.predict(X_test)

In [10]:
from sklearn.metrics import r2_score, mean_absolute_error

In [11]:
print("R2_score :",r2_score(y_test, y_pred))
print("MAE :",mean_absolute_error(y_test, y_pred))

R2_score : 0.668759493535632
MAE : 3.189091965887848


## 🧪 3. Exploring Ridge Alpha Values
Regularization is controlled by the hyperparameter **Alpha**. 
* **Small Alpha (0.1):** Behaves like Linear Regression.
* **Large Alpha (100):** Heavily shrinks coefficients to simplify the model.

In [12]:
from sklearn.linear_model import Ridge

In [13]:
rr = Ridge(alpha=0.1)
rr.fit(X_train,y_train)
print("Coef_ : ", rr.coef_)
print("\n Intercept : ",rr.intercept_)

Coef_ :  [-1.12399694e-01  3.04593914e-02  3.48958400e-02  2.75033318e+00
 -1.59244585e+01  4.44577949e+00 -7.30474388e-03 -1.42960751e+00
  2.60042840e-01 -1.07802286e-02 -9.00771040e-01  1.24004789e-02
 -5.10902332e-01]

 Intercept :  29.36627127257693


In [14]:
rr = Ridge(alpha=10)
rr.fit(X_train,y_train)
print("Coef_ : ", rr.coef_)
print("\n Intercept : ",rr.intercept_)

Coef_ :  [-0.10713363  0.03555248 -0.02627747  1.81329133 -1.88924475  4.19532572
 -0.01534126 -1.23262135  0.24803063 -0.01274419 -0.76176896  0.01283334
 -0.561835  ]

 Intercept :  22.439732052474042


In [15]:
rr = Ridge(alpha=100)
rr.fit(X_train,y_train)
print("Coef_ : ", rr.coef_)
print("\n Intercept : ",rr.intercept_)

Coef_ :  [-1.10764853e-01  3.98919010e-02 -4.86253730e-02  5.50701733e-01
 -1.97858825e-01  2.43881473e+00  5.45476646e-04 -1.12939994e+00
  2.99013586e-01 -1.46298901e-02 -8.17852407e-01  1.19512041e-02
 -6.89539142e-01]

 Intercept :  34.62659588633344


## 🏗️ 4. Building Ridge Regression from Scratch
We implement the **Normal Equation** version of Ridge Regression. The objective is to solve for $W$ (Weights) using:
$$W = (X^T X + \alpha I)^{-1} X^T y$$

### 🛠️ Technical Details:
* **Bias Integration:** We insert a column of ones into the training data to handle the intercept ($b$) automatically.
* **Identity Matrix Adjustment:** Crucially, we set $I[0,0] = 0$ because we do **not** want to regularize the intercept.
* **Vectorization:** We use NumPy's matrix multiplication (`dot`) and inversion (`inv`) for high-speed calculation.

In [16]:
class Myridge:
    def __init__(self,alpha = 0.1):
        self.alpha = alpha
        self.coef_ = None
        self.intercept_ = None
    
    def fit(self, X_train, y_train):
        X_train = np.insert(X_train, 0, 1, axis=1)
        I = np.identity(X_train.shape[1])
        I[0,0] = 0
        W = np.linalg.inv(np.dot(X_train.T, X_train) + self.alpha*I).dot(X_train.T).dot(y_train)

        self.coef_ = W[1:]
        self.intercept_ = W[0] 
        
    def predict(self, X_test):
        return np.dot(X_test, self.coef_) + self.intercept_

## 📊 5. Verification & Comparison
We now instantiate our custom `Myridge` class and compare its performance against the results from Scikit-Learn. If our math is correct, the coefficients and $R^2$ scores should be nearly identical.

In [17]:
reg = Myridge()
reg.fit(X_train, y_train)
y_pred1 = reg.predict(X_test)
print(r2_score(y_test, y_pred1))
print("MAE :",mean_absolute_error(y_test, y_pred1))

0.6686244122020883
MAE : 3.1788559832342944


## ⚖️ 6. Feature Scaling & Final Validation
Ridge Regression is sensitive to the scale of features because the L2 penalty is applied to the magnitude of the coefficients. 

To ensure a fair penalty across all 13 features (e.g., comparing 'CRIM' to 'TAX'), we apply **Standardization** ($z = (x - \mu) / \sigma$).

### Observations:
* By scaling the data, our scratch model's $R^2$ score now perfectly aligns with the professional `sklearn` implementation.
* The coefficients are now on a comparable scale, making the model more stable and interpretable.

In [18]:
scalar = StandardScaler()
X_train_scaled =  scalar.fit_transform(X_train)
X_test_scaled =  scalar.transform(X_test)

In [19]:
reg = Myridge()
reg.fit(X_train_scaled, y_train)
y_pred1 = reg.predict(X_test_scaled)
print(r2_score(y_test, y_pred1))
print("MAE :",mean_absolute_error(y_test, y_pred1))

0.6687298368808314
MAE : 3.1887231092563364


In [20]:
print(reg.coef_)
print(reg.intercept_)

[-1.00111591  0.69436316  0.27539404  0.71912548 -2.01912122  3.14590087
 -0.17617627 -3.07816919  2.24333232 -1.75959591 -2.03674427  1.12933027
 -3.61037565]
22.79653465346535


## 🏁 Conclusion
* **Accuracy:** The scratch model successfully achieved an $R^2$ of **0.6686**, matching the professional libraries.
* **Key Insight:** Building models from scratch reveals that Regularization is simply a mathematical "stabilizer" added to the covariance matrix.